# 🚀 Notebook 04: Clasificación con Modelos de Boosting — Google Colab

**Tema Asignado:** Boosting: Gradient Boost, XGBoost y Ada Boost

En esta fase (el estudio principal exigido por la rúbrica), abordamos el problema desde una perspectiva de **Clasificación**. El objetivo es predecir si un caso será de **larga duración** (mayor a 1 año) o no, utilizando estrictamente los tres modelos de ensamble asignados.

Esto nos permite evaluar el rendimiento con métricas de clasificación (Accuracy, F1-Score, Curva ROC, Matriz de Confusión), logrando un cumplimiento perfecto de la rúbrica de la materia.

In [ ]:
# Instalar dependencias
!pip install -q xgboost

In [ ]:
# ============================================================
# 📂 SUBIR ARCHIVO PROCESADO
# ============================================================
# Sube el archivo: casos_procesados.csv
# (Generado por el notebook 02_preprocessing)
# Si ya lo corriste en esta sesión, el archivo ya estará disponible.
# ============================================================
from google.colab import files
import os

os.makedirs('/content/data/processed', exist_ok=True)
os.makedirs('/content/images', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)

csv_path = '/content/data/processed/casos_procesados.csv'
if not os.path.exists(csv_path):
    print("⬆️ Sube el archivo 'casos_procesados.csv' (generado en notebook 02)")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(f'/content/{filename}', csv_path)
        print(f"✅ Archivo movido a: {csv_path}")
else:
    print(f"✅ Archivo ya existe en: {csv_path}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Cargar Datos Procesados
df = pd.read_csv('/content/data/processed/casos_procesados.csv')
print(f"Total de registros: {df.shape[0]}")
df.head()

In [ ]:
# 2. Preparar Variables (X e y)
# Usaremos 'caso_largo' como target de clasificación.
X = df.drop(['duracion_dias', 'caso_largo', 'delito', 'hecho_departamento'], axis=1) # Usaremos numéricas y luego dummies

# One-hot encoding simple para categóricas
X_cat = pd.get_dummies(df[['delito', 'hecho_departamento']], drop_first=True)
X = pd.concat([X, X_cat], axis=1)

y = df['caso_largo']

# Split Train/Test 70/30
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Escalado de variables numéricas (GradientBoost y AdaBoost no lo requieren estrictamente, pero ayuda a la convergencia)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Set de Entrenamiento: {X_train_scaled.shape[0]} registros")
print(f"Set de Prueba: {X_test_scaled.shape[0]} registros")

## 3. Función de Evaluación

In [ ]:
def evaluar_clasificador(nombre_modelo, modelo, X_train, X_test, y_train, y_test):
    print(f"\n--- Entrenando {nombre_modelo} ---")
    start_time = time.time()
    modelo.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"AUC-ROC:  {roc_auc:.4f}")
    print(f"Tiempo de entrenamiento: {train_time:.2f}s")
    
    return {
        'Modelo': nombre_modelo,
        'Accuracy': acc,
        'F1-Score': f1,
        'AUC-ROC': roc_auc,
        'Tiempo(s)': train_time,
        'FPR': fpr,
        'TPR': tpr,
        'CM': cm
    }

In [ ]:
# 4. Implementación de los Modelos de Boosting
resultados = []

# 4.1 Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
res_gb = evaluar_clasificador("Gradient Boost", gb, X_train_scaled, X_test_scaled, y_train, y_test)
resultados.append(res_gb)

# 4.2 XGBoost
xgb = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', random_state=42)
res_xgb = evaluar_clasificador("XGBoost", xgb, X_train_scaled, X_test_scaled, y_train, y_test)
resultados.append(res_xgb)

# 4.3 AdaBoost
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
res_ada = evaluar_clasificador("Ada Boost", ada, X_train_scaled, X_test_scaled, y_train, y_test)
resultados.append(res_ada)

## 5. Comparación y Visualizaciones

In [ ]:
# Curva ROC Comparativa
plt.figure(figsize=(10, 8))
for res in resultados:
    plt.plot(res['FPR'], res['TPR'], lw=2, label=f"{res['Modelo']} (AUC = {res['AUC-ROC']:.3f})")

plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curvas ROC - Comparación de Modelos de Boosting')
plt.legend(loc="lower right")
plt.savefig('/content/images/curva_roc_boosting.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrices de Confusión
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, res in enumerate(resultados):
    sns.heatmap(res['CM'], annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f"Matriz de Confusión: {res['Modelo']}")
    axes[i].set_xlabel('Predicción')
    axes[i].set_ylabel('Real')
plt.tight_layout()
plt.savefig('/content/images/matrices_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Guardar resultados finales
import IPython.display as display_mod
df_res = pd.DataFrame(resultados).drop(['FPR', 'TPR', 'CM'], axis=1)
df_res.to_csv('/content/results/resultados_clasificacion_boosting.csv', index=False)
print("\n🏆 Tabla de Resultados Finales:")
display_mod.display(df_res)

In [ ]:
# Descargar todos los resultados e imágenes
from google.colab import files as colab_files
import glob

print("📥 Descargando resultados e imágenes...")
colab_files.download('/content/results/resultados_clasificacion_boosting.csv')

for img in glob.glob('/content/images/*.png'):
    print(f"📥 Descargando: {img}")
    colab_files.download(img)

print("\n✅ ¡Todo descargado!")